In [ ]:
import os
import certifi
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.agents import create_react_agent , AgentExecutor
import streamlit
import langsmith
from langchain.tools import tool


In [ ]:

##env variables
load_dotenv()
gemeni_api_key = os.getenv("GEMENI_API_KEY")
travily_api_key = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")



In [ ]:
##Tools
search_tool = TavilySearchResults(max_results=4)
tools = [search_tool]

@tool
def get_weather_data(city :str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )
    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )



In [113]:

##LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temprature=1.0,
    google_api_key= gemeni_api_key

)


In [114]:
##prompt
prompt = hub.pull("hwchase17/react")


/home/calvin/Downloads/yes/envs/langagnet/lib/python3.11/site-packages/langchain/hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [115]:
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)


In [116]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [118]:
response = agent_executor.invoke({
    "input": (
        "Find the capital of Egypt"
    )
})



> Entering new AgentExecutor chain...
Thought: The capital of Egypt is well-known general knowledge.
Final Answer: The capital of Egypt is Cairo.

> Finished chain.
